In [ ]:
pip install datasets optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 23.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/data/"

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv(path+"output.csv")
print(len(df))
df = df[df['price_change_4h'].notna()]
df = df[df['volume_surge_price'].notna()]
print(len(df))
df

20258
20251


,timestamp,title,source,volume_surge_price,price_change_4h
0,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru,-1.006608e-06,-0.000297
1,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru,-2.814638e-03,-0.013606
2,1630005294,Record breaking sale for Bored Ape,Watcher Guru,6.691295e-03,0.013199
3,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru,-9.948841e-07,0.000483
4,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru,2.904352e-04,0.000779
...,...,...,...,...,...
20246,1743146885,NY AG files $200 million settlement with Galax...,unfolded.,-1.440233e-02,-0.009280
20247,1743169126,Terraform Labs to open crypto creditor claims ...,unfolded.,-7.996360e-02,-0.010471
20248,1743181914,FDIC clarifies that banks can engage in permit...,unfolded.,-2.301976e-02,-0.006169
20249,1743188025,Trump pardons three co-founders of BitMex cryp...,unfolded.,-2.509775e-04,0.000081


Init and modify FinBERT for regression

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

class QuenRegression(nn.Module):
    def __init__(self, model_name="WiroAI/WiroAI-Finance-Qwen-1.5B"):
        super(QuenRegression, self).__init__()
        self.quen = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
        #self.regressor = nn.Linear(10, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.quen(input_ids=input_ids, attention_mask=attention_mask)
        #regression_score = self.regressor(outputs.logits)
        normalized_score = torch.tanh(outputs.logits)
        return normalized_score


class AdaptiveWeightedHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(AdaptiveWeightedHuberLoss, self).__init__()
        self.delta = delta

    def forward(self, predictions, targets, weights=None):
        error = predictions - targets
        abs_error = torch.abs(error)
        quadratic = torch.minimum(abs_error, torch.tensor(self.delta))
        linear = abs_error - quadratic
        loss = 0.5 * quadratic ** 2 + self.delta * linear
        if weights is not None:
            loss = loss * weights
        return loss.mean()







In [ ]:
# Load model & tokenizer
model = QuenRegression()
tokenizer = AutoTokenizer.from_pretrained("WiroAI/WiroAI-Finance-Qwen-1.5B")
model.quen.config.pad_token_id = tokenizer.pad_token_id

for param in model.quen.base_model.parameters():
    param.requires_grad = False

# Unfreeze only the last N transformer blocks (e.g., last 2 layers)
# You may need to inspect the model architecture — here's a generic approach:
encoder_layers = model.quen.base_model.layers  # Common path for Qwen/ChatGLM-like models

for layer in encoder_layers[-8:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in model.quen.score.parameters():
    param.requires_grad = True

# Make sure classifier head stays trainable (if you want it fine-tuned)
# for param in model.quen.classifier.parameters():
#     param.requires_grad = True

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.quen.gradient_checkpointing_enable()
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at WiroAI/WiroAI-Finance-Qwen-1.5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/6.75k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

QuenRegression(
  (quen): Qwen2ForSequenceClassification(
    (model): Qwen2Model(
      (embed_tokens): Embedding(151936, 1536)
      (layers): ModuleList(
        (0-27): 28 x Qwen2DecoderLayer(
          (self_attn): Qwen2Attention(
            (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
            (k_proj): Linear(in_features=1536, out_features=256, bias=True)
            (v_proj): Linear(in_features=1536, out_features=256, bias=True)
            (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
          )
          (mlp): Qwen2MLP(
            (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
            (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
            (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
          (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
   

In [ ]:
model = torch.compile(model)

In [ ]:
# text = "Billions must live."
# inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
# inputs = {k: v.to(device) for k, v in inputs.items()}

# with torch.no_grad():
#     output = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

# predicted_score = output.squeeze().item()
# print(f"Predicted sentiment score: {predicted_score:.4f}")

Data split

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from datetime import timedelta

#torch.cuda.empty_cache()

class SentimentDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_length=128):
        self.texts = texts
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        score = self.scores[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(score, dtype=torch.float).unsqueeze(0)  # Ensure correct shape
        }

# train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
# print(len(train_df))

# train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
# train_loader = DataLoader(train_set, batch_size=16, shuffle=True)

# test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
# test_loader = DataLoader(test_set, batch_size=16, shuffle=False)


df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Sort by timestamp just in case
df = df.sort_values('timestamp')

# Define the cutoff date for 365 days before the latest timestamp
cutoff_date = df['timestamp'].max() - timedelta(days=365)
print(df['timestamp'].max())
print(cutoff_date)

# Split the data
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

# Print sizes for sanity check
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Create datasets and dataloaders
train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)

test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

2025-03-28 21:11:59
2024-03-28 21:11:59
Train size: 17570, Test size: 2681


In [ ]:
# import optuna
# from torch.utils.data import DataLoader

# def objective(trial):
#     # Search space
#     optimizer_name = "SGD" # trial.suggest_categorical("optimizer", ["AdamW", "SGD", "RMSprop"])
#     lr = trial.suggest_float("lr", 0.00001, 0.0006, log=True)
#     batch_size = 24 # trial.suggest_categorical("batch_size", [8, 16])
#     max_len = 128 # trial.suggest_categorical("max_len", [64, 128])
#     weight_decay = trial.suggest_float("weight_decay", 0.02, 0.065)
#     momentum = trial.suggest_float("momentum", 0.25, 0.5)

#     # Prepare datasets with trial-specific max_len
#     df_small = df.sample(n=1000, random_state=42)
#     train_df_small, test_df_small = train_test_split(df_small, test_size=0.1, random_state=42)
#     print(len(train_df))

#     train_set_small = SentimentDataset(train_df_small["title"].tolist(), train_df_small["volume_surge_price"].tolist(), tokenizer, max_length=max_len)
#     train_loader_small = DataLoader(train_set_small, batch_size=16, shuffle=True)

#     test_set_small = SentimentDataset(test_df_small["title"].tolist(), test_df_small["volume_surge_price"].tolist(), tokenizer, max_length=max_len)
#     test_loader_small = DataLoader(test_set_small, batch_size=16, shuffle=False)

#     # Reinitialize model
#     model = QuenRegression()
#     model.quen.config.pad_token_id = tokenizer.pad_token_id

#     for param in model.quen.base_model.parameters():
#         param.requires_grad = False
#     for layer in model.quen.base_model.layers[-8:]:
#         for param in layer.parameters():
#             param.requires_grad = True

#     if optimizer_name == "AdamW":
#         optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#     elif optimizer_name == "Adam":
#         optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
#     elif optimizer_name == "SGD":
#         optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=momentum)
#     elif optimizer_name == "RMSprop":
#         optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)

#     model.to(device)

#     loss_fn = AdaptiveWeightedHuberLoss()

#     # Train for 1 epoch (for speed)
#     model.train()
#     for batch in train_loader:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()
#         outputs = model(input_ids, attention_mask)
#         loss = loss_fn(outputs.squeeze(-1), labels.squeeze(-1))
#         loss.backward()
#         optimizer.step()

#     # Evaluate
#     preds, labels = evaluate(model, test_loader_small, device)
#     metrics = compute_metrics(preds, labels, test_df, metric_label="trial")

#     return metrics["MSE"]  # Minimize MSE

# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20)

# print("\n🏆 Best Trial:")
# print(study.best_trial)


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np



def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            predictions = outputs.squeeze().cpu().numpy()

            loss = F.mse_loss(outputs.squeeze(), labels.squeeze())
            total_loss += loss.item()
            #predictions = [predictions]
            predictions = np.array(predictions)

            if predictions.ndim == 0:
                predictions = [predictions.item()]  # Convert scalar to list
            else:
                predictions = predictions.tolist()

            all_preds.extend(predictions)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    print(f"Test MSE Loss: {avg_loss:.4f}")

    return all_preds, all_labels

In [ ]:


def compute_metrics(predictions, actuals, df, metric_label="Model"):
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    if len(predictions) == 0 or len(actuals) == 0:
        print(f"{metric_label} - Skipping metrics computation: empty predictions or actuals.")
        return {
            "Metric": metric_label,
            "MSE": None,
            "RMSE": None,
            "MAE": None,
            "MAPE": None,
            "SMAPE": None,
            "R²": None,
            "Corr": None,
            "Corr_P": None,
        }

    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)

    epsilon = 1e-8  # Avoid division by zero
    mape = np.mean(np.abs((actuals - predictions) / (actuals + epsilon))) * 100
    smape = 100 * np.mean(2.0 * np.abs(predictions - actuals) / (np.abs(predictions) + np.abs(actuals) + epsilon))
    r2 = r2_score(actuals, predictions)

    target_correlation = df.loc[df.index[:len(predictions)], 'price_change_4h'].values
    corr_with_price_change_4h, corr_p_val = pearsonr(predictions, target_correlation)

    print(f"{metric_label} Performance:")
    print(f"🔹 MSE:   {mse:.4f}")
    print(f"🔹 RMSE:  {rmse:.4f}")
    print(f"🔹 MAE:   {mae:.4f}")
    print(f"🔹 MAPE:  {mape:.2f}%")
    print(f"🔹 SMAPE: {smape:.2f}%")
    print(f"🔹 R²:    {r2:.4f}")
    print(f"🔹 Corr with price_change_4h: {corr_with_price_change_4h:.4f}")
    print(f"🔹 Corr p: {corr_p_val:.4f}")
    print("-" * 40)

    return {
        "Metric": metric_label,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "SMAPE": smape,
        "R²": r2,
        f"Corr": corr_with_price_change_4h,
        f"Corr_P": corr_p_val,
    }


train

In [ ]:
from tqdm.notebook import tqdm
from torch.amp import autocast, GradScaler

torch.set_float32_matmul_precision('high')
torch.cuda.empty_cache()
scaler = GradScaler('cuda')

def train(model, train_loader, optimizer, loss_fn, device, epochs=20):

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        loop = tqdm(train_loader, leave=False)
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with autocast('cuda'):  # AMP context
                outputs = model(input_ids, attention_mask)
                loss = loss_fn(outputs.squeeze(-1), labels.squeeze(-1))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")
        predictions_finetuned, actuals = evaluate(model, test_loader, device)
        metrics_finetuned = compute_metrics(predictions_finetuned,actuals, test_df, "Fine-Tuned Qwen")
        torch.cuda.empty_cache()

#optimizer = torch.optim.SGD(model.parameters(), lr=0.0001, weight_decay=0.045, momentum=0.35)
#optimizer = torch.optim.SGD(model.parameters(), lr=0.0006, weight_decay=0.012, momentum=0.4)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00006, weight_decay=0.042)

loss_fn = AdaptiveWeightedHuberLoss()
train(model, train_loader, optimizer, loss_fn, device, epochs=20)


  0%|          | 0/1099 [00:00<?, ?it/s]

Epoch 1: Loss = 0.0184
Test MSE Loss: 0.0098
Fine-Tuned Qwen Performance:
🔹 MSE:   0.0098
🔹 RMSE:  0.0988
🔹 MAE:   0.0784
🔹 MAPE:  80409.08%
🔹 SMAPE: 168.03%
🔹 R²:    -0.6108
🔹 Corr with price_change_4h: 0.0666
🔹 Corr p: 0.0006
----------------------------------------


  0%|          | 0/1099 [00:00<?, ?it/s]

Epoch 2: Loss = 0.0019
Test MSE Loss: 0.0062
Fine-Tuned Qwen Performance:
🔹 MSE:   0.0062
🔹 RMSE:  0.0786
🔹 MAE:   0.0302
🔹 MAPE:  8040.57%
🔹 SMAPE: 160.15%
🔹 R²:    -0.0183
🔹 Corr with price_change_4h: 0.0353
🔹 Corr p: 0.0679
----------------------------------------


  0%|          | 0/1099 [00:00<?, ?it/s]

Epoch 3: Loss = 0.0017
Test MSE Loss: 0.0061
Fine-Tuned Qwen Performance:
🔹 MSE:   0.0061
🔹 RMSE:  0.0779
🔹 MAE:   0.0310
🔹 MAPE:  10919.83%
🔹 SMAPE: 156.48%
🔹 R²:    -0.0011
🔹 Corr with price_change_4h: 0.0621
🔹 Corr p: 0.0013
----------------------------------------


... (22 outputs left out)

  0%|          | 0/1099 [00:00<?, ?it/s]

Epoch 15: Loss = 0.0003
Test MSE Loss: 0.0064
Fine-Tuned Qwen Performance:
🔹 MSE:   0.0064
🔹 RMSE:  0.0799
🔹 MAE:   0.0391
🔹 MAPE:  25213.98%
🔹 SMAPE: 157.12%
🔹 R²:    -0.0535
🔹 Corr with price_change_4h: 0.0627
🔹 Corr p: 0.0012
----------------------------------------


  0%|          | 0/1099 [00:00<?, ?it/s]

KeyboardInterrupt: 

evaluate

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np

# train_df, test_df = train_test_split(df, test_size=0.1, random_state=13)
# train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
# train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
# test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
# test_loader = DataLoader(test_set, batch_size=8, shuffle=False)


def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            predictions = outputs.squeeze().cpu().numpy()

            loss = F.mse_loss(outputs.squeeze(), labels.squeeze())
            total_loss += loss.item()

            all_preds.extend(predictions)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    print(f"Test MSE Loss: {avg_loss:.4f}")

    return all_preds, all_labels

def compute_metrics(predictions, actuals, df, metric_label="Model"):
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)

    epsilon = 1e-8  # Avoid division by zero
    mape = np.mean(np.abs((actuals - predictions) / (actuals + epsilon))) * 100
    smape = 100 * np.mean(2.0 * np.abs(predictions - actuals) / (np.abs(predictions) + np.abs(actuals) + epsilon))
    r2 = r2_score(actuals, predictions)

    target_correlation = df.loc[df.index[:len(predictions)], 'price_change_4h'].values
    corr_with_price_change_4h, p_value = pearsonr(predictions, target_correlation)

    print(f"{metric_label} Performance:")
    print(f"🔹 MSE:   {mse:.4f}")
    print(f"🔹 RMSE:  {rmse:.4f}")
    print(f"🔹 MAE:   {mae:.4f}")
    print(f"🔹 MAPE:  {mape:.2f}%")
    print(f"🔹 SMAPE: {smape:.2f}%")
    print(f"🔹 R²:    {r2:.4f}")
    print(f"🔹 Corr with price_change_4h: {corr_with_price_change_4h:.4f}")
    print(f"🔹 Corr p-value: {p_value:.4e}")  # Scientific notation for clarity
    print("-" * 40)

    return {
        "Metric": metric_label,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "SMAPE": smape,
        "R²": r2,
        f"Corr": corr_with_price_change_4h,
        "Corr_p_value": p_value
    }



In [ ]:
# Run evaluation
predictions_finetuned, actuals = evaluate(model, test_loader, device)
print(predictions_finetuned)
print(actuals)
metrics_finetuned = compute_metrics(predictions_finetuned,actuals, test_df, "Fine-Tuned Quen")

In [ ]:
torch.save(model.state_dict(), 'model_weights_quen.pth')

In [ ]:
#torch.save(model.state_dict(), path+'model_weights_quen.pth')
from google.colab import drive
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/data/"

class QuenRegression(nn.Module):
    def __init__(self, model_name="WiroAI/WiroAI-Finance-Qwen-1.5B"):
        super().__init__()
        self.quen = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(self.quen.config.hidden_size, 1),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.quen(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        raw_score = self.regressor(cls_output)
        normalized_score = torch.tanh(raw_score)
        return normalized_score


model = QuenRegression()
tokenizer = AutoTokenizer.from_pretrained("WiroAI/WiroAI-Finance-Qwen-1.5B")
model.quen.config.pad_token_id = tokenizer.pad_token_id

for param in model.quen.base_model.parameters():
    param.requires_grad = False

encoder_layers = model.quen.base_model.layers  # Common path for Qwen/ChatGLM-like models

for layer in encoder_layers[-8:]:
    for param in layer.parameters():
        param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
state_dict = torch.load(path+'model_weights_quen.pth')

# If the state_dict keys are prefixed with '_orig_mod', remove it before loading
# This is the typical case when saving a compiled model's state_dict
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('_orig_mod.'):
        new_key = k[len('_orig_mod.'):]
        new_state_dict[new_key] = v
    else:
        new_state_dict[k] = v

# Load the potentially modified state_dict into the uncompiled model
#model.load_state_dict(new_state_dict)
model.eval()  # set to eval mode if you're not training
tokenizer = AutoTokenizer.from_pretrained("WiroAI/WiroAI-Finance-Qwen-1.5B")


In [ ]:
# # Load model & tokenizer
model2 = QuenRegression()
model2.quen.config.pad_token_id = tokenizer.pad_token_id

model2.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2.to(device)
# predictions2, actuals2 = evaluate(model2, test_loader, device)
# metrics_base = compute_metrics(predictions2, actuals2, test_df, "Base Quen Fin")

In [ ]:
import pandas as pd

df = pd.read_csv(path+"BTCUSDT.csv", sep='|')
df = df.rename(columns={'1502942400': 'timestamp', '4261.48': 'open', '4261.48.1': 'high', '4261.48.2': 'low', '4261.48.3': 'close', '1.775183': 'btc_volume_sell', '320.390851': 'usd_volume_buy', '0.075183': 'btc_volume_buy', '7564.906851': 'usd_volume_sell', '3': 'activity'})

df

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

def backtest_sentiment_trading(
    model,
    tokenizer,
    df,
    text_data,
    initial_portfolio_usd=100_000,
    trade_fraction=0.5,
    hold_period=4,
    sentiment_treshold=0.2,
    minutes=525600
):
    model.eval()
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    text_data['timestamp'] = pd.to_datetime(text_data['timestamp'], unit='s')

    df = df.sort_values('timestamp')
    text_data = text_data.sort_values('timestamp')

    start_time = df['timestamp'].max() - timedelta(minutes=minutes)
    df = df[df['timestamp'] >= start_time]
    text_data = text_data[text_data['timestamp'] >= start_time]

    portfolio_value = initial_portfolio_usd
    trades = []

    for _, row in text_data.iterrows():
        timestamp = row['timestamp']
        text = row['title']

        # Tokenize and run sentiment model
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            sentiment = model(
                input_ids=inputs["input_ids"].cuda(),
                attention_mask=inputs["attention_mask"].cuda()
            ).item()

        if abs(sentiment) < sentiment_treshold:
            continue  # No trade if sentiment is neutral

        #print(sentiment)

        entry_row = df[df['timestamp'] >= timestamp].head(1)
        exit_row = df[df['timestamp'] >= timestamp + timedelta(hours=hold_period)].head(1)

        if entry_row.empty or exit_row.empty:
            continue

        entry_price = entry_row['close'].values[0]
        exit_price = exit_row['close'].values[0]

        trade_usd = portfolio_value * trade_fraction
        btc_amount_to_trade = trade_usd / entry_price

        if sentiment > sentiment_treshold:
            pnl = btc_amount_to_trade * (exit_price - entry_price)  # Long
            action = "BUY"
        else:
            pnl = btc_amount_to_trade * (entry_price - exit_price)  # Short
            action = "SELL"

        portfolio_value += pnl

        trades.append({
            'time': timestamp,
            'action': action,
            'sentiment': sentiment,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'btc_amount': btc_amount_to_trade,
            'pnl': pnl,
            'portfolio_value': portfolio_value
        })

    trades_df = pd.DataFrame(trades)

    # Save individual trades to CSV
    trades_df.to_csv(path + "trades_quen_trained_all.csv", index=False)

    # Plot only if there are trades
    if not trades_df.empty:
        trades_df['cumulative_pnl'] = trades_df['portfolio_value'] - initial_portfolio_usd

        plt.figure(figsize=(12, 6))
        plt.plot(trades_df['time'], trades_df['cumulative_pnl'], marker='o', linestyle='-')
        plt.title("Cumulative Profit and Loss Over Time")
        plt.xlabel("Date")
        plt.ylabel("Cumulative PnL (USDT)")
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print(f"Final Portfolio Value: {portfolio_value:.2f} USDT")
        print(f"Total PnL: {portfolio_value - initial_portfolio_usd:.2f} USDT from {len(trades_df)} trades")
    else:
        print("No trades executed during the backtest period.")

    return trades_df


In [ ]:
news_df = pd.read_csv(path+"output.csv")

trades = backtest_sentiment_trading(
    model=model,
    tokenizer=tokenizer,
    df=df,
    text_data=news_df,  # contains 'timestamp' and 'title'
    hold_period=2,
    sentiment_treshold=0.05,
    minutes=60*24*365  #365 # 1 year
)

In [ ]:
news_df = pd.read_csv(path+"output.csv")

trades = backtest_sentiment_trading(
    model=model2,
    tokenizer=tokenizer,
    df=df,
    text_data=news_df,  # contains 'timestamp' and 'title'
    hold_period=2,
    sentiment_treshold=0.1,
    minutes=60*24*365  # 1 year
)